In [46]:
import os
import gc
import shutil
import traceback
from typing import List, Tuple
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.lines import Line2D
from tqdm import tqdm
from itertools import combinations

# Optional RSS probe - only used to confirm memory stays flat during long runs
try:
    import psutil
    _PROC = psutil.Process()
except Exception:
    _PROC = None

## Parameters

In [47]:
variables = []
with open("00 Variables.txt", "r") as f:
    lines = f.readlines()
    hashes = 1
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            hashes = len(s) - len(s.lstrip("#"))
            comment_indent = (hashes - 1) * 2
            print(" " * comment_indent + "\033[92m" + s + "\033[0m")
        elif "=" in s:
            var = s.split("=", 1)[0].strip()
            variables.append(var)
            try:
                exec(s)
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                value_part = s.split("#", 1)[0].strip()
                comment_part = s[s.find("#"):] if "#" in s else ""
                eval_value = f"{var} = {repr(eval(var))}"
                if comment_part:
                    print(" " * black_indent + eval_value + " \033[38;5;250m" + "  " + comment_part + "\033[0m")
                else:
                    print(" " * black_indent + eval_value)
            except Exception as e:
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                print(" " * black_indent + s)

# Global Settings
  TICKER = '^GSPC'
  START_DATE = '2024-09-01'
  END_DATE = '2026-04-30'
  WINDOW_SIZE = 126
  REGIME_WINDOW = 126   # Lookback for regime stats (vol, quantile bands). Keep = WINDOW_SIZE so bands match the label
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05
  MAX_WINDOWS = 10000
  DATE_INFO = True
  INDICATOR_SIZE = 'small'   # "small", "large"
  CHART_NUM = 4   # 9 panels split as [3, 2, 2, 2]
  TRANSPARENT = False
  CHART_DPI = 72   # 144 = original; 72 halves each edge and quarters figure memory
  NEWS_QUERY = 'Standard and Poors 500 stock index'
  MODEL_NAME = 'gpt-5-mini'   # "gpt-5-nano-2025-08-07", "gpt-5-mini-2025-08-07", "gpt-5-2025-08-07"
  EMBEDDING_MODEL = 'text-embedding-3-large'
  RUN_TIMESTAMP = '20260603 1232'   # This will be changed automatically
  N_PREV = 6   # Number of previous windows to provide context foBr
  USE_IMAGES = True   # If True, send images (multimodal model), else just text
# Path
  ## Layer 1
     L1_RESULTS = 'Results'
     L1_COMB

## Definition

In [48]:
def clean_folder(folder_path: str):
    print(f"[INFO] Cleaning folder: {folder_path}")
    if os.path.exists(folder_path):
        deleted_any = False
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                    deleted_any = True
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                    deleted_any = True
            except Exception as e:
                print(f"[ERROR] Failed to delete {file_path}. Reason: {e}\n")
        if deleted_any:
            print(f"[DELETE] Folder cleaned: {folder_path}\n")
        else:
            print(f"[INFO] Folder exists but is already clean: {folder_path}\n")
    else:
        os.makedirs(folder_path, exist_ok=True)
        print(f"[CREATE] Folder created: {folder_path}\n")

### Each Panel's Definition

In [49]:
# ---------------------------------------------------------------------------
# Shared styling. Factored out of the panels so every chart is formatted
# identically - the panels below then contain only the plotting decisions.
#
# Panel budget: at most 3 plotted DATA SERIES per panel. Three things do not count
# against it, because none adds a quantity the reader has to track separately:
#   - shaded percentile bands (Bollinger, vol 5-95)
#   - constant reference lines (0, 30/70, ADX 25, +/-2 sigma)
#   - event markers annotating a series already plotted (death/golden cross sit on
#     the close line; tail breaches highlight points of the return scatter)
# So panels 1 and 5 show 4 legend entries but only 3 series each.
#
# Colour convention (by lookback where a panel spans horizons, by quantity where
# it does not):
#   blue   = fast horizon      orange = medium horizon     black = slow horizon
#   purple = structural (200d) red    = risk / downside    grey   = bands, neutral
# Reference-line grammar:
#   neutral baseline -> thin solid grey     decision threshold -> dashed
#   percentile range -> fill_between at low alpha, never a line
#   twin-axis series -> dashed, so which axis a line belongs to is readable
# ---------------------------------------------------------------------------

def _style_y_axis(ax, label):
    ax.set_ylabel(label, color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight('bold')


def _style_x_axis(ax, x_vals, xticks, xticklabels):
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for lbl in ax.get_xticklabels():
        lbl.set_fontweight('bold')
        lbl.set_rotation(0)

    if DATE_INFO == True:
        pad = pd.Timedelta(days=2)
    else:
        pad = int(len(x_vals) * 0.05) or 1
    ax.set_xlim([x_vals[0] - pad, x_vals[-1] + pad])
    ax.grid(True, linestyle='--', alpha=0.3)


def _style_legend(ax, loc="upper left"):
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return
    ax.legend(handles, labels, loc=loc, fontsize=18,
              prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)


def _baseline(ax, y=0):
    ax.axhline(y, color='black', linestyle='-', lw=1.5, alpha=0.25)


# ---------------------------------------------------------------------------
# Panel 1 - Where is price relative to its structural trend, and has the
#           trend regime just flipped?
# ---------------------------------------------------------------------------
def plot_panel_1(ax, window, x_vals, xticks, xticklabels):
    # Rebased to 100 at the window start so this axis means the same thing in
    # every one of the ~289 windows, which is what makes windows comparable.
    #
    # sma20 is deliberately absent: it is the Bollinger midline (already implied
    # by the band) and panel 8's z-score measures stretch from it directly.
    base = window['close'].iloc[0]
    close_r = window['close'].values / base * 100

    ax.fill_between(x_vals,
                    window['bb_upper'].values / base * 100,
                    window['bb_lower'].values / base * 100,
                    alpha=0.12, color="grey", label="Bollinger 20d")

    ax.plot(x_vals, close_r, label="Close", color="black", lw=3)
    ax.plot(x_vals, window['sma50'].values / base * 100,
            label="SMA50", lw=3, linestyle="--", color="orange")
    ax.plot(x_vals, window['sma200'].values / base * 100,
            label="SMA200", lw=3, linestyle="--", color="purple")

    _baseline(ax, 100)

    # Both crossing averages are plotted above, so a cross marker has a visible cause.
    for flag, color, marker, lbl in [("death_cross", "red", "v", "Death Cross"),
                                     ("golden_cross", "gold", "^", "Golden Cross")]:
        idx = [i for i, val in enumerate(window[flag]) if val == 1]
        if idx:
            ax.scatter([x_vals[i] for i in idx], close_r[idx],
                       color=color, marker=marker, s=200, label=lbl, zorder=5)

    _style_y_axis(ax, "Price (Window Start = 100)")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 2 - Is volatility outside its own regime, and is the shock fresh?
# ---------------------------------------------------------------------------
def plot_panel_2(ax, window, x_vals, xticks, xticklabels):
    # Three horizons of ONE measure - all three are now close-to-close log-return
    # vol, so the SPREAD between them is the signal: hv_10 far above the 126d line
    # means a shock the regime has not yet absorbed.
    #
    # The 126d line and its band share 125 of 126 observations with the chart
    # window, so they move slowly by construction. That is their job here: they
    # are the regime anchor the fast horizons are read against, not a shape.
    ax.fill_between(x_vals, window['volatility_5pct'].values, window['volatility_95pct'].values,
                    alpha=0.12, color="grey", label="Vol 5-95 pct (126d)")

    for name, color, lbl in [("hv_10", "blue", "HV 10d"),
                             ("hv_60", "orange", "HV 60d"),
                             ("volatility", "black", "Volatility 126d")]:
        ax.plot(x_vals, window[name].values, label=lbl, lw=3, color=color)

    _style_y_axis(ax, "Annualized Volatility")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 3 - Is the daily range expanding out of a compression regime?
# ---------------------------------------------------------------------------
def plot_panel_3(ax, window, x_vals, xticks, xticklabels):
    # Squeeze -> expansion is the classic pre-break setup, which needs one fast
    # and one slow range measure. Both are scale-free, so a single % axis works.
    # Replaces atr, which was in index points and so not comparable across windows.
    #
    # bb_width dominates the axis (typically 4-10% against tr_percent's 0.5-2%),
    # which is the honest consequence of sharing units. A twin axis for two
    # quantities already in the same units would be worse.
    ax.plot(x_vals, window['tr_percent'].values, label="True Range 1d", lw=3, color="blue")
    # bb_width is a fraction of sma20; x100 puts it in the same % units as tr_percent.
    ax.plot(x_vals, window['bb_width'].values * 100, label="Bollinger Width 20d", lw=3, color="black")

    _baseline(ax)
    _style_y_axis(ax, "Range (% of Close)")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 4 - Is the return distribution turning left-skewed and fat-tailed?
# ---------------------------------------------------------------------------
def plot_panel_4(ax, window, x_vals, xticks, xticklabels):
    # Twin axis is genuinely warranted: skewness spans about +/-1 while excess
    # kurtosis spans 0-10.
    #
    # Single horizon on purpose. The 20-day pair that used to share this panel was
    # deleted upstream: at n=20 the standard error of skewness is ~0.55 and of
    # excess kurtosis ~1.10, so both moved with whichever single outlier happened
    # to be inside the window. Colour here separates the two quantities rather
    # than two horizons.
    ax.plot(x_vals, window['skewness_60'].values, label="Skewness 60d", lw=3, color="black")
    _baseline(ax)

    ax2 = ax.twinx()
    ax2.plot(x_vals, window['kurtosis_60'].values, label="Excess Kurtosis 60d",
             lw=3, linestyle="--", color="orange")

    _style_y_axis(ax, "Skewness")
    _style_y_axis(ax2, "Excess Kurtosis")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)
    _style_legend(ax2, loc="upper right")


# ---------------------------------------------------------------------------
# Panel 5 - Are returns breaching their own recent tail, and how deep is it?
# ---------------------------------------------------------------------------
def plot_panel_5(ax, window, x_vals, xticks, xticklabels):
    # The deeper the tail, the longer the window it needs: the 5% level at a
    # quarter, the 1% level at the regime window. The 20-day VaR/ES columns were
    # deleted upstream - at n=20 the 5% "quantile" was just the 2nd smallest
    # return and "ES" was the rolling minimum.
    #
    # The crash label's own boundary is deliberately NOT drawn here. It is
    # r_5pct.shift(1), and plotting the unshifted r_5pct drew a line that was not
    # the boundary the label applies while marking breaches the label never fired
    # on. Breach markers below key off the 5% VaR line, which IS plotted, so every
    # marker has a visible cause.
    logret = window['logreturn'].values
    var_5 = window['var_5pct_60'].values

    ax.scatter(x_vals, logret, s=28, alpha=0.45, color="grey", label="Daily Log Return")

    mask = logret <= var_5
    if mask.any():
        ax.scatter([x for x, m in zip(x_vals, mask) if m], logret[mask],
                   s=110, color="red", marker="x", lw=3, label="Tail Breach", zorder=5)

    ax.plot(x_vals, var_5, label="5% VaR 60d", lw=3, color="orange")
    ax.plot(x_vals, window['r_1pct'].values, label="1% VaR 126d", lw=3, color="red")

    _baseline(ax)
    _style_y_axis(ax, "Log Return")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 6 - How deep is the drawdown, and how persistent has the pain been?
# ---------------------------------------------------------------------------
def plot_panel_6(ax, window, x_vals, xticks, xticklabels):
    # Depth against depth-and-duration. Both are already in percent, so one axis
    # serves - no twin needed. Signs follow convention: drawdown is negative,
    # Ulcer Index is positive, so they open away from the zero baseline.
    #
    # dd_20 replaces maxdd_20, which nested rolling(20).max() inside
    # rolling(20).min() and so reached back up to 39 days despite its name.
    ax.plot(x_vals, window['dd_20'].values, label="Drawdown from 20d High", lw=3, color="red")
    ax.plot(x_vals, window['ulcer_index'].values, label="Ulcer Index 14d", lw=3, color="black")

    _baseline(ax)
    _style_y_axis(ax, "Drawdown / Pain (%)")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 7 - Is price stretched, and is this a trending or a choppy move?
# ---------------------------------------------------------------------------
def plot_panel_7(ax, window, x_vals, xticks, xticklabels):
    # stoch_k used to share this panel with rsi, but the two are ~0.8 correlated:
    # both are 14-day range-position measures, so the second one added pixels
    # rather than information. ADX answers a different question - trend strength,
    # not position - and is bounded on the same 0-100 scale, so one axis still
    # serves. ADX rarely exceeds 60, so it sits in the lower half by nature.
    x_start, x_end = x_vals[0], x_vals[-1]

    ax.plot(x_vals, window['rsi'].values, label="RSI 14d", lw=3, color="blue")
    ax.plot(x_vals, window['adx'].values, label="ADX 14d", lw=3, color="black")

    ax.plot([x_start, x_end], [70, 70], color="red", linestyle="--", lw=3, alpha=0.6, label="Overbought (70)")
    ax.plot([x_start, x_end], [30, 30], color="green", linestyle="--", lw=3, alpha=0.6, label="Oversold (30)")
    ax.plot([x_start, x_end], [25, 25], color="grey", linestyle=":", lw=3, alpha=0.6, label="ADX Trending (25)")

    # Fixed limits: a bounded indicator should not rescale between windows.
    ax.set_ylim(-5, 105)
    _style_y_axis(ax, "Oscillator (0-100)")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)


# ---------------------------------------------------------------------------
# Panel 8 - How large is the move, and how many sigma from its own mean?
# ---------------------------------------------------------------------------
def plot_panel_8(ax, window, x_vals, xticks, xticklabels):
    # One ROC horizon only: roc_5, roc_10 and roc_20 are strongly collinear, so
    # the extra two lines added no information. ROC is in percent and therefore
    # comparable across windows.
    #
    # The twin axis carries zscore: distribution-relative stretch in sigma, a
    # different question from panel 7's range-relative position.
    ax.plot(x_vals, window['roc_20'].values, label="ROC 20d", lw=3, color="black")
    _baseline(ax)

    ax2 = ax.twinx()
    ax2.plot(x_vals, window['zscore'].values, label="Z-Score 20d", lw=3, linestyle="--", color="red")
    ax2.plot([x_vals[0], x_vals[-1]], [2, 2], color="red", linestyle=":", lw=3, alpha=0.6, label="+2 Sigma")
    ax2.plot([x_vals[0], x_vals[-1]], [-2, -2], color="red", linestyle=":", lw=3, alpha=0.6, label="-2 Sigma")

    _style_y_axis(ax, "Momentum / ROC (%)")
    _style_y_axis(ax2, "Z-Score (Std Dev)")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)
    _style_legend(ax2, loc="upper right")


# ---------------------------------------------------------------------------
# Panel 9 - Is the move backed by volume and flow, or is it thin?
# ---------------------------------------------------------------------------
def plot_panel_9(ax, window, x_vals, xticks, xticklabels):
    # Liquidity/flow had no panel at all despite cmf being computed, so a move on
    # no volume looked identical to one on a volume spike.
    #
    # Twin axis: cmf is bounded to [-1, 1] while volume_z_20 routinely reaches +4
    # on a shock, so sharing one axis would flatten cmf into the baseline. obv is
    # excluded - its level depends on an arbitrary cumulative origin, so it is not
    # comparable between windows; a volume z-score is.
    #
    # cmf keeps a FIXED [-1, 1] axis for the same reason panel 7 pins 0-100: a
    # bounded indicator must not rescale between windows. In practice it uses only
    # the middle of that range, so it is also filled to the zero baseline - that
    # makes the accumulation/distribution sign readable at low amplitude without
    # rescaling the axis or clipping an extreme reading.
    cmf = window['cmf'].values
    ax.fill_between(x_vals, 0, cmf, alpha=0.25, color="black")
    ax.plot(x_vals, cmf, label="Chaikin Money Flow 20d", lw=3, color="black")
    ax.set_ylim(-1.05, 1.05)
    _baseline(ax)

    ax2 = ax.twinx()
    ax2.plot(x_vals, window['volume_z_20'].values, label="Volume Z-Score 20d",
             lw=3, linestyle="--", color="blue")
    ax2.plot([x_vals[0], x_vals[-1]], [2, 2], color="blue", linestyle=":", lw=3, alpha=0.6, label="+2 Sigma")

    _style_y_axis(ax, "Money Flow (-1 to 1)")
    _style_y_axis(ax2, "Volume Z-Score")
    _style_x_axis(ax, x_vals, xticks, xticklabels)
    _style_legend(ax)
    _style_legend(ax2, loc="upper right")

In [50]:
def create_chart(window, ticker, date_str, panels, chart_index, save_fig_dir, _transparent=TRANSPARENT, dpi=CHART_DPI):
    chart_folder = os.path.join(save_fig_dir, f"Chart {chart_index}")
    os.makedirs(chart_folder, exist_ok=True)
    filename = f"{ticker}_{date_str}_chart{chart_index}.png"
    filepath = os.path.join(chart_folder, filename)

    # Pyplot-free figure: never registered in the global figure registry (Gcf),
    # so it is released when it goes out of scope instead of waiting on
    # plt.close() plus a cyclic GC pass. This is what keeps memory flat over
    # the many hundreds of figures a full run produces.
    fig = Figure(figsize=(40, 7 * len(panels)), dpi=dpi)
    FigureCanvasAgg(fig)  # attach an Agg canvas explicitly
    try:
        axs = fig.subplots(len(panels), 1, squeeze=False)[:, 0]

        if DATE_INFO == True:
            x_vals = window.index
            xticks = window.index[::max(1, len(window) // 10)]
            xticklabels = [d.strftime("%Y-%m-%d") for d in xticks]
        else:
            x_vals = list(range(len(window)))
            xticks = list(range(0, len(window), max(1, len(window)//10)))
            xticklabels = [str(i) for i in xticks]

        for i, panel_func in enumerate(panels):
            panel_func(axs[i], window, x_vals, xticks, xticklabels)

        title_str = f"{ticker} - {date_str.replace('-', '')}" if DATE_INFO else f"{ticker}"
        fig.suptitle(title_str, fontsize=28, fontweight='bold', y=0.995)

        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        fig.savefig(filepath, transparent=_transparent, dpi=dpi)
    finally:
        fig.clear()  # drop artists even if a panel raised

    return filepath


# Nine panels, each answering one distinct anomaly-detection question with at most three
# plotted series. Panel numbers match the rendered order - no panel is conditional, so
# code panel N == chart panel N.
#   1 trend & regime       2 volatility shock vs regime   3 range & squeeze
#   4 distribution shape   5 tail risk                    6 drawdown & pain
#   7 stretch vs trend     8 momentum & sigma stretch     9 flow confirmation
# Module level rather than rebuilt per window: it is fixed data, and validate_indicator_set
# below needs it before any rendering starts.
PANEL_FUNCS = [
    plot_panel_1, plot_panel_2, plot_panel_3, plot_panel_4, plot_panel_5,
    plot_panel_6, plot_panel_7, plot_panel_8, plot_panel_9,
]

PANEL_REQUIREMENTS = {
    plot_panel_1: ["close", "sma50", "sma200", "bb_upper", "bb_lower", "death_cross", "golden_cross"],
    plot_panel_2: ["hv_10", "hv_60", "volatility", "volatility_5pct", "volatility_95pct"],
    plot_panel_3: ["tr_percent", "bb_width"],
    plot_panel_4: ["skewness_60", "kurtosis_60"],
    plot_panel_5: ["logreturn", "var_5pct_60", "r_1pct"],
    plot_panel_6: ["dd_20", "ulcer_index"],
    plot_panel_7: ["rsi", "adx"],
    plot_panel_8: ["roc_20", "zscore"],
    plot_panel_9: ["cmf", "volume_z_20"],
}


def validate_indicator_set(df_ind):
    """
    Every panel is mandatory. A missing column used to drop its panel behind a [WARN] and
    then re-split the remainder across the output files, so "chart2.png" stopped meaning
    the same thing between runs. These images are model input, so a change in their shape
    has to be an error.

    Called once before rendering (and before the output folder is cleaned) so a stale or
    mismatched indicator CSV fails fast instead of failing identically 289 times.
    """
    available = set(df_ind.columns)
    absent = {
        f"panel {i + 1}": [col for col in PANEL_REQUIREMENTS[pf] if col not in available]
        for i, pf in enumerate(PANEL_FUNCS)
    }
    absent = {panel: cols for panel, cols in absent.items() if cols}
    if absent:
        raise KeyError(
            f"Indicator set is missing required columns: {absent}. "
            "indicators_list_small (notebook 01) and PANEL_REQUIREMENTS (here) must stay "
            "in sync - re-run notebook 01 if you changed either."
        )
    print(f"[OK] All {len(PANEL_FUNCS)} panels have their columns ({len(available)} available)")


def create_technical_chart(window, ticker, date_str, save_fig_dir="figures", chart_num=CHART_NUM):
    n_panels = len(PANEL_FUNCS)
    panels_per_chart = [n_panels // chart_num] * chart_num
    for i in range(n_panels % chart_num):
        panels_per_chart[i] += 1

    charts = []
    start = 0
    for i in range(chart_num):
        end = start + panels_per_chart[i]
        if start >= n_panels:
            break
        chart = create_chart(window, ticker, date_str, PANEL_FUNCS[start:end], chart_index=i + 1, save_fig_dir=save_fig_dir)
        charts.append(chart)
        start = end

    return tuple(charts)

In [51]:
def generate_rolling_window_charts_only(df_ind, window_size, save_fig_dir, max_windows=MAX_WINDOWS, ticker=TICKER,
                                       gc_every=5):
    # Validate BEFORE clean_folder: a stale or mismatched indicator CSV should fail
    # without having already deleted the previous run's charts.
    validate_indicator_set(df_ind)

    # Always rebuild from scratch: deletes the Chart 1..N folders and their PNGs,
    # and recreates save_fig_dir if absent. create_chart re-makes each "Chart {i}"
    # subfolder as it saves. This guarantees no stale chart from an older panel
    # definition can survive a re-run.
    clean_folder(save_fig_dir)

    first_valid_index = df_ind.first_valid_index()
    first_valid_pos = df_ind.index.get_loc(first_valid_index)
    start_pos = first_valid_pos + window_size
    N = min(len(df_ind), start_pos + max_windows)
    total = N - start_pos
    failures = 0

    # The first window_size rows are consumed building the first window, so charts start
    # ~126 trading days after START_DATE (~2025-03 for START_DATE=2024-09-01) rather than
    # at START_DATE itself. That is intentional: every window is then built exclusively
    # from in-period data.
    if total <= 0:
        raise ValueError(
            f"{len(df_ind)} row(s) available but {window_size} are needed for one window. "
            "Widen [START_DATE, END_DATE] or reduce WINDOW_SIZE."
        )
    print(f"[INFO] {total} window(s): {df_ind.index[start_pos - 1]:%Y-%m-%d} to {df_ind.index[N - 1]:%Y-%m-%d}")

    with tqdm(total=total, desc="Generating Rolling Charts") as pbar:
        for i in range(start_pos, N):
            window = df_ind.iloc[i - window_size: i]
            date_str = window.index[-1].strftime("%Y-%m-%d")

            try:
                image_paths = create_technical_chart(window, ticker, date_str, save_fig_dir=save_fig_dir)
                pbar.set_postfix_str(f"Saved {ticker}_{date_str} Series Plot")
            except Exception as e:
                failures += 1
                print(f"[ERROR] Chart failed for {date_str}: {type(e).__name__}: {e}")
                # Full traceback once only - 289 tracebacks would bloat the notebook
                # and choke the frontend, which is its own way of losing the kernel.
                if failures == 1:
                    traceback.print_exc()

            # Periodic cleanup + leak probe. open_figs should always be 0 now that
            # create_chart bypasses pyplot; anything else means pyplot crept back in.
            # Each window allocates ~58 MB of canvas at CHART_DPI=72, and matplotlib
            # artists sit in reference cycles, so collecting every 5 windows caps the
            # uncollected backlog at ~290 MB instead of ~1.4 GB.
            if gc_every and (i - start_pos + 1) % gc_every == 0:
                gc.collect()
                open_figs = len(plt.get_fignums())
                if open_figs:
                    print(f"[WARN] {open_figs} pyplot figure(s) still open at {date_str}")
                if _PROC is not None:
                    print(f"[MEM] {date_str}: RSS {_PROC.memory_info().rss / 1e9:.2f} GB")

            pbar.update(1)

    print(f"[DONE] {total - failures} window(s) rendered, {failures} failed -> {save_fig_dir}")

## Main Execution

In [52]:
FIG_DIR = f"{L1_RESULTS}/{L2_FIGURES}/{L3_FIGURES_CHARTS}"
# No folder handling needed here: generate_rolling_window_charts_only cleans and
# recreates FIG_DIR itself on every run.

print(f"Inputing price data for {TICKER} from {START_DATE} to {END_DATE} ...")
df_prices = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv", index_col=0, parse_dates=True)
print("Data inputted. Shape:", df_prices.shape, "\n")

print("Inputing indicators...")
df_indicators = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}.csv", index_col=0, parse_dates=True)
print("Indicators inputed. Shape:", df_indicators.shape, "\n")

print("Generating rolling window charts (starting from first day with full indicator data)...")

Inputing price data for ^GSPC from 2024-09-01 to 2026-04-30 ...
Data inputted. Shape: (808, 5) 

Inputing indicators...
Indicators inputed. Shape: (557, 59) 

Generating rolling window charts (starting from first day with full indicator data)...


In [53]:
print(f"Inputing price data for {TICKER} from {START_DATE} to {END_DATE} ...")
df = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv", index_col=0, parse_dates=True)
df = df[(df.index >= START_DATE) & (df.index <= END_DATE)]
print("Data inputted. Shape:", df.shape, "\n")

print("Inputing indicators...")
if INDICATOR_SIZE == "small":
    INDICATORS_NAME = INDICATORS + "_" + INDICATOR_SIZE
else:
    INDICATORS_NAME = INDICATORS

df_indicators = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_NAME}.csv", index_col=0, parse_dates=True)
df_indicators = df_indicators[(df_indicators.index >= START_DATE) & (df_indicators.index <= END_DATE)]
print("Indicators inputed. Shape:", df_indicators.shape, "\n")

print("Inputing crash labels data...")
df_crash = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{CRASH_LABEL}.csv", index_col=0, parse_dates=True)
df_crash = df_crash[(df_crash.index >= START_DATE) & (df_crash.index <= END_DATE)]
print("Crash labels data inputed. Shape:", df_crash.shape, "\n")

print("Inputing news data...")
df_news = pd.read_csv(f"{L1_RESULTS}/{L2_NEWS}/{NEWS_CACHE}.csv", index_col=0, parse_dates=True)
df_news = df_news[(df_news.index >= START_DATE) & (df_news.index <= END_DATE)]
print("News data inputed. Shape:", df_news.shape, "\n")

Inputing price data for ^GSPC from 2024-09-01 to 2026-04-30 ...
Data inputted. Shape: (415, 5) 

Inputing indicators...
Indicators inputed. Shape: (415, 27) 

Inputing crash labels data...
Crash labels data inputed. Shape: (415, 3) 

Inputing news data...
News data inputed. Shape: (607, 1) 



In [54]:
generate_rolling_window_charts_only(
    df_ind=df_indicators,
    window_size=WINDOW_SIZE,
    save_fig_dir=FIG_DIR,
    max_windows=MAX_WINDOWS,
    ticker=TICKER
)

[OK] All 9 panels have their columns (27 available)
[INFO] Cleaning folder: Results/Figures/Generate Charts
[DELETE] Folder cleaned: Results/Figures/Generate Charts

[INFO] 289 window(s): 2025-03-05 to 2026-04-29


Generating Rolling Charts:   2%|▏         | 5/289 [00:05<05:25,  1.14s/it, Saved ^GSPC_2025-03-11 Series Plot]

[MEM] 2025-03-11: RSS 0.25 GB


Generating Rolling Charts:   3%|▎         | 10/289 [00:11<05:09,  1.11s/it, Saved ^GSPC_2025-03-18 Series Plot]

[MEM] 2025-03-18: RSS 0.27 GB


...

[MEM] 2025-06-20: RSS 0.26 GB


Generating Rolling Charts:  28%|██▊       | 80/289 [01:32<03:43,  1.07s/it, Saved ^GSPC_2025-06-27 Series Plot]

[MEM] 2025-06-27: RSS 0.27 GB


Generating Rolling Charts:  28%|██▊       | 82/289 [01:35<04:01,  1.17s/it, Saved ^GSPC_2025-07-01 Series Plot]


KeyboardInterrupt: 